# Peaky Blinders Downloader to Google Drive
Download Peaky Blinders episodes directly to your Google Drive using Colab.

In [ ]:
# ==========================
# 1. Install dependencies
# ==========================
!pip install tqdm requests --quiet

In [ ]:
import os
import requests
import threading
import time
import urllib.parse
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

In [ ]:
# ==========================
# 2. Mount Google Drive (workaround for credential bug)
# ==========================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==========================
# 3. Set up your Drive folder
# ==========================
drive_folder = '/content/drive/MyDrive/PeakyBlinders'
os.makedirs(drive_folder, exist_ok=True)

In [ ]:
# ==========================
# 4. List of download URLs
# ==========================
urls = [
    "https://filetolink.netabots.com/AgADfB41019/Peaky%20Blinders%20S01E01%20720p%2010bit%20BluRay%20x265%20HEVC%20%5BOrg%20TV-DL.mkv",
    "https://filetolink.netabots.com/AgAD4h41017/Peaky%20Blinders%20S01E02%20720p%2010bit%20BluRay%20x265%20HEVC%20%5BOrg%20Zee%20C.mkv",
    "https://filetolink.netabots.com/AgAD_B41020/Peaky%20Blinders%20S01E03%20720p%2010bit%20BluRay%20x265%20HEVC%20%5BOrg%20Zee%20C.mkv",
    "https://filetolink.netabots.com/AgADYh41016/Peaky%20Blinders%20S01E04%20720p%2010bit%20BluRay%20x265%20HEVC%20%5BOrg%20Zee%20C.mkv",
    "https://filetolink.netabots.com/AgADXR41015/Peaky%20Blinders%20S01E05%20720p%2010bit%20BluRay%20x265%20HEVC%20%5BOrg%20Zee%20C.mkv",
    "https://filetolink.netabots.com/AgAD5B41018/Peaky%20Blinders%20S01E06%20720p%2010bit%20BluRay%20x265%20HEVC%20%5BOrg%20Zee%20C.mkv"
]

In [ ]:
# ==========================
# 5. Download & Upload Functions
# ==========================
def format_size(size_bytes):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if size_bytes < 1024.0 or unit == 'GB':
            break
        size_bytes /= 1024.0
    return f"{size_bytes:.2f} {unit}"

def get_original_filename(url):
    encoded_filename = url.split('/')[-1]
    original_filename = urllib.parse.unquote(encoded_filename)
    return original_filename

def download_file(url, output_path, position):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        start_time = time.time()
        with open(output_path, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, 
            desc=f"DL {os.path.basename(output_path)[:30]}...", 
            ncols=100, position=position*2+1, leave=True,
            bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
                    elapsed = time.time() - start_time
                    speed = pbar.n / elapsed if elapsed > 0 else 0
                    eta = (total - pbar.n) / speed if speed > 0 else 0
                    pbar.set_postfix({
                        'Speed': f"{speed/1024/1024:.2f} MB/s",
                        'Elapsed': f"{elapsed:.1f}s",
                        'ETA': f"{eta:.1f}s",
                        'Size': format_size(total)
                    }, refresh=True)
        total_time = time.time() - start_time
    return total_time

def upload_to_drive(src_path, dst_folder, position):
    file_name = os.path.basename(src_path)
    dst_path = os.path.join(dst_folder, file_name)
    total = os.path.getsize(src_path)
    start_time = time.time()
    with open(src_path, 'rb') as src_file, open(dst_path, 'wb') as dst_file, tqdm(
        total=total, unit='B', unit_scale=True, 
        desc=f"UP {file_name[:30]}...", 
        ncols=100, position=position*2+2, leave=True,
        bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
    ) as pbar:
        while True:
            chunk = src_file.read(1024*1024)
            if not chunk:
                break
            dst_file.write(chunk)
            pbar.update(len(chunk))
            elapsed = time.time() - start_time
            speed = pbar.n / elapsed if elapsed > 0 else 0
            eta = (total - pbar.n) / speed if speed > 0 else 0
            pbar.set_postfix({
                'Speed': f"{speed/1024/1024:.2f} MB/s",
                'Elapsed': f"{elapsed:.1f}s",
                'ETA': f"{eta:.1f}s",
                'Size': format_size(total)
            }, refresh=True)
    total_time = time.time() - start_time
    return total_time, dst_path

def process_file(url, position):
    original_filename = get_original_filename(url)
    print(f"\nProcessing: {original_filename}\n")
    dl_time = download_file(url, original_filename, position)
    print(f"Download completed in {dl_time:.2f} seconds.")
    up_time, drive_path = upload_to_drive(original_filename, drive_folder, position)
    print(f"Upload completed in {up_time:.2f} seconds.")
    print(f"File uploaded to Google Drive at: {drive_path}\n")
    os.remove(original_filename)

In [ ]:
# ==========================
# 6. Run All Tasks in Parallel (Professional Boxed Output)
# ==========================
print("=" * 80)
print(f"Starting download of {len(urls)} Peaky Blinders episodes to Google Drive")
print(f"Target folder: {drive_folder}")
print("=" * 80)

max_concurrent = 6  # Adjust for Colab resources

with ThreadPoolExecutor(max_workers=max_concurrent) as executor:
    futures = [executor.submit(process_file, url, i) for i, url in enumerate(urls)]
    for future in futures:
        future.result()

print("\n" + "=" * 80)
print("All files have been downloaded and uploaded to your Google Drive folder:")
print(drive_folder)
print("=" * 80)